# uchoice bakery hyperedge prediction split

Load the uchoice_bakery hypergraph and run the train/test hyperedge split three times with different random seeds.

In [1]:
from pathlib import Path
import pickle

import numpy as np

from hypergraphx import Hypergraph
from src.hypergraph_train_test_split import (
    split_hypergraph_train_test,
    generate_negative_hyperedges,
)

In [2]:
data_path = Path("data") / "uchoice_bakery" / "uchoice_bakery.pickle"
with open(data_path, "rb") as f:
    payload = pickle.load(f)

edge_list = payload["edge_list"]
H = Hypergraph(edge_list)

print(f"Loaded: {data_path}")
print(f"Edge list entries (paper-style columns): {len(edge_list)}")
print(f"Hypergraphx unique |E|: {len(H.get_edges())}")
print(f"Hypergraphx |V|: {len(H.get_nodes())}")

Loaded: data/uchoice_bakery/uchoice_bakery.pickle
Edge list entries (paper-style columns): 63896
Hypergraphx unique |E|: 27035
Hypergraphx |V|: 50


In [ ]:
test_ratio = 0.2
seeds = list(range(10))

splits_path = Path("data") / "uchoice_bakery" / "splits.pkl"
splits_path.parent.mkdir(parents=True, exist_ok=True)

splits = []
if splits_path.exists():
    with open(splits_path, "rb") as f:
        saved = pickle.load(f)

    if saved.get("test_ratio") == test_ratio and saved.get("seeds") == seeds:
        print(f"Loading cached splits from {splits_path}")
        for item in saved["splits"]:
            H_train = Hypergraph(item["train_edge_list"])
            splits.append(
                {
                    "seed": item["seed"],
                    "H_train": H_train,
                    "test_edges": item["test_edges"],
                    "negative_edges": item["negative_edges"],
                }
            )
    else:
        print("Cached splits do not match current test_ratio/seeds; recomputing.")

if not splits:
    print("Computing splits and saving cache...")
    saved_splits = []

    for seed in seeds:
        print(f"\n=== Split with seed={seed} ===")
        H_train, test_edges = split_hypergraph_train_test(H, test_ratio=test_ratio, seed=seed)
        negative_edges = generate_negative_hyperedges(H, test_edges, seed=seed)

        split_item = {
            "seed": seed,
            "H_train": H_train,
            "test_edges": test_edges,
            "negative_edges": negative_edges,
        }
        splits.append(split_item)

        saved_splits.append(
            {
                "seed": seed,
                "train_edge_list": list(H_train.get_edges()),
                "test_edges": test_edges,
                "negative_edges": negative_edges,
            }
        )

        print(f"Train |E| (unique): {len(H_train.get_edges())}")
        print(f"Held-out positive edges: {len(test_edges)}")
        print(f"Held-out negative edges: {len(negative_edges)}")

    cache_payload = {
        "test_ratio": test_ratio,
        "seeds": seeds,
        "splits": saved_splits,
    }
    with open(splits_path, "wb") as f:
        pickle.dump(cache_payload, f, protocol=pickle.HIGHEST_PROTOCOL)
    print(f"Saved splits cache: {splits_path}")

Computing splits and saving cache...

=== Split with seed=0 ===


In [5]:
from itertools import product
from pathlib import Path
import pandas as pd

from src.utils import hmrw_embeddings, hyper2vec_embeddings
from src.hypergraph_train_test_split import evaluate

pqr_grid = [
    # Local / sticky
    [0.25, 4, 1],
    [0.5, 2, 1],

    # Baseline
    [1, 1, 1],

    # Structured exploration (favour overlap)
    [4, 4, 1],

    # Exploratory
    [2, 0.5, 1],
    [4, 0.25, 1],

    # r variation (node-level effect)
    [1, 1, 0.5],
    [1, 1, 2],
]

num_walks = 5
walk_length = 100

out_path = Path("results") / "contact" / "walks.csv"
out_path.parent.mkdir(parents=True, exist_ok=True)

rows = []
if out_path.exists():
    rows = pd.read_csv(out_path).to_dict("records")

existing = {
    (
        int(row["split_seed"]),
        float(row["p"]),
        float(row["q"]),
        float(row["r"]),
        int(row["num_walks"]),
        int(row["walk_length"]),
    )
    for row in rows
}

for split in splits:
    seed = split["seed"]
    H_train = split["H_train"]
    test_edges = split["test_edges"]
    negative_edges = split["negative_edges"]

    nodes_train = sorted(H_train.get_nodes())

    for p, q, r in pqr_grid:
        key = (int(seed), float(p), float(q), float(r), int(num_walks), int(walk_length))
        if key in existing:
            print(f"skip existing split={seed}, p={p}, q={q}, r={r}")
            continue

        X = hmrw_embeddings(
            H_train,
            p=p,
            q=q,
            r=r,
            num_walks=num_walks,
            walk_length=walk_length,
            seed=seed,
        )

        emb = {node: X[i] for i, node in enumerate(nodes_train)}
        auc, ap = evaluate(test_edges, negative_edges, emb)

        row = {
            "split_seed": seed,
            "p": p,
            "q": q,
            "r": r,
            "num_walks": num_walks,
            "walk_length": walk_length,
            "auc": auc,
            "ap": ap,
            "n_test_pos": len(test_edges),
            "n_test_neg": len(negative_edges),
        }
        rows.append(row)
        existing.add(key)

        # Save after every iteration so progress is not lost.
        pd.DataFrame(rows).to_csv(out_path, index=False)
        print(
            f"saved split={seed}, p={p}, q={q}, r={r}, "
            f"auc={auc:.4f}, ap={ap:.4f} -> {out_path}"
        )

results_df = pd.DataFrame(rows)
results_df.tail()

KeyboardInterrupt: 

In [7]:
from src.utils import hmrw_embeddings, hyper2vec_embeddings, clique_node2vec_embeddings

In [6]:
xx = hyper2vec_embeddings(
    H_train,
    p=1,
    q=0.25,
    num_walks=5,
    walk_length=40,
    seed=seed,
)

emb = {node: xx[i] for i, node in enumerate(nodes_train)}
auc, ap = evaluate(test_edges, negative_edges, emb)

print(
    f"auc={auc:.4f}, ap={ap:.4f} -> {out_path}"
)

auc=0.9238, ap=0.9317 -> results/contact/walks.csv


In [8]:
xx = hmrw_embeddings(
    H_train,
    p=1,
    q=0.25,
    num_walks=5,
    walk_length=40,
    seed=seed,
)

emb = {node: xx[i] for i, node in enumerate(nodes_train)}
auc, ap = evaluate(test_edges, negative_edges, emb)

print(
    f"auc={auc:.4f}, ap={ap:.4f} -> {out_path}"
)

Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


auc=0.9199, ap=0.9314 -> results/contact/walks.csv


In [9]:
xx = clique_node2vec_embeddings(
    H_train,
    p=1,
    q=0.25,
    num_walks=5,
    walk_length=40,
    seed=seed,
)

emb = {node: xx[i] for i, node in enumerate(nodes_train)}
auc, ap = evaluate(test_edges, negative_edges, emb)

print(
    f"auc={auc:.4f}, ap={ap:.4f} -> {out_path}"
)

Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


auc=0.9313, ap=0.9307 -> results/contact/walks.csv
